In [57]:
import os
os.environ["OLLAMA_HOST"] = "http://localhost:11460" 
print(os.environ.get("OLLAMA_HOST"))

http://localhost:11460


In [58]:
pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [59]:
import prompts as prompts


In [60]:
import importlib
importlib.reload(prompts)

<module 'prompts' from '/home/bumu60du/fake_review_detection/prompts.py'>

In [61]:
import numpy as np
import pandas as pd
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

In [62]:
from prompts import Config

In [63]:
# Load your CSV
df_new = pd.read_csv("/home/bumu60du/fake_review_detection/Datasets/Amazon_Human_VS_ComputerFake.csv")
reviews = df_new["text"].tolist()

In [64]:
prompt = PromptTemplate(
    input_variables=["text"], # the placeholder to be replaced based on prompt template
    template=Config.zero_shot_prompt_template
)

# Load small model
llm = OllamaLLM(model=Config.model)

# Create chain using pipe operator (modern LangChain syntax)
chain = prompt | llm

In [65]:
print(Config.zero_shot_prompt_template)


    You are an expert at detecting fake and real reviews.

    Task: Classify the following review of a product that is listed on amazon as either "fake" or "real". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording.

    Review: "{text}"

    Classify this review as either "fake" or "real" (respond with only one word):


In [66]:
# # Classify each review
# results = []
# for r in reviews:
#     result = chain.invoke({"text": r})
#     # Clean the output - extract only "truthful" or "deceptive"
#     label = result.strip().lower()
#     # Extract first word if model adds extra text
#     if " " in label:
#         label = label.split()[0]
#     # Remove any punctuation
#     label = label.strip('.,!?;:')
#     print(f"Raw output: {result} -> Cleaned: {label}")
#     results.append(label)

In [67]:
# Classify each review
results = []

for r in reviews:
    result = chain.invoke({"text": r})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        print(f"Skipping invalid output: {result}")
        continue  # skip this review
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    results.append(cleaned_label)


Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real.

The review provides specific and descriptive feedback on the product's appearance, construction, and color. It also suggests that the reviewer plans to purchase the product as a gift, indicating a genuine interest in the product. -> Cleaned: real
Raw output: Real.

The review uses natural language and expresses genuine enthusiasm for the product. -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real.

The review is concise and descriptive, indicating genuine experience with the product. -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real

In [68]:
df_new.head(10)


,Binary_label,Category,source,text,is_synthetic
0,fake,Home_and_Kitchen,amazon,Air pressure was utter crap. I kept the lid o...,1
1,fake,Home_and_Kitchen,amazon,"Not what I expected, smaller than expected, bu...",1
2,fake,Home_and_Kitchen,amazon,We like a toaster that doesn't have the long h...,1
3,fake,Home_and_Kitchen,amazon,Love these pans they work great. The only prob...,1
4,fake,Home_and_Kitchen,amazon,I love the print on the back and the finish. I...,1
5,fake,Home_and_Kitchen,amazon,Bought this on a whim and it was the best purc...,1
6,fake,Home_and_Kitchen,amazon,"I admit I was leary of the suction, but I did ...",1
7,fake,Home_and_Kitchen,amazon,Purchased one as a gift and it was a very nice...,1
8,fake,Home_and_Kitchen,amazon,I wanted something to hold a bunch of pieces o...,1
9,fake,Home_and_Kitchen,amazon,I'm so happy I found this set and am very plea...,1


In [69]:
# def clean_label(text):
#     return text.strip().lower().split()[0].strip('.,!?;:')
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def evaluate_model(df, results):
    # Get true labels from the dataframe
    #y_true = df["deceptive"].tolist()
    y_true = df["Binary_label"].tolist()
    y_pred = results

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', pos_label=None)
    conf_matrix = confusion_matrix(y_true, y_pred)

    return accuracy, f1, conf_matrix

In [70]:
y_true = df_new["Binary_label"].tolist()

In [71]:
for label in results:
    print(label)

real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
fake
real
real
real
real
real
real
real
real
real
real
fake
real
real
real
real
real
real
real
real
real
real
real
real
fake
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
fake
real
real
real
real
real
real
real
fake
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
real
fake
real
real
real
real
real
real
real
fake
real
real
real
real
real
real
real
fake
real
real
real
real
real
real


In [72]:
accuracy, f1, conf_matrix = evaluate_model(df_new, results)

print("=" * 50)
print("ZERO-SHOT PROMPTING RESULTS")
print(Config.model)
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

ZERO-SHOT PROMPTING RESULTS
gemma:7b
Accuracy: 0.51875

F1 Score: 0.4040236145967775

Confusion Matrix:
[[ 32 368]
 [ 17 383]]


## One-Shot Prompting
Using one example to guide the model's classification

In [73]:
llm = OllamaLLM(model=Config.model)

In [74]:
one_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template = Config.one_shot_prompt_template
)
# Create one-shot chain
one_shot_chain = one_shot_prompt | llm

In [75]:
print(Config.one_shot_prompt_template)


     You are an expert at detecting fake and real reviews.

    Task: Classify the following review of a product that is listed on amazon as either "real" or "fake". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording and lacks details or specificity.

    Example:
    Review: "Great Watch. I have had this watch for a long time and it works great. I have one in my car and the other in my office. The watch itself is a great watch and is very easy to use. It is a perfect size for my wrist and is very easy to wear. I have used it with a portable watch and I can tell the watch is very well made. I have worn it with a watch band and it is very comfortable. I will buy another one for my phone. I would highly recommend this watch.Perfect for my kids. I love them.These are super lightweight and lightweight. I have used them for a couple of months and they are still working great. I would definitely recomme

In [76]:
# # Classify each review
# one_shot_results = []
# for r in reviews:
#     result = one_shot_chain.invoke({"text": r})
#     # Clean the output - extract only "truthful" or "deceptive"
#     label = result.strip().lower()
#     # Extract first word if model adds extra text
#     if " " in label:
#         label = label.split()[0]
#     # Remove any punctuation
#     label = label.strip('.,!?;:')
#     print(f"Raw output: {result} -> Cleaned: {label}")
#     one_shot_results.append(label)

In [77]:
# Classify each review
one_shot_results = []

for r in reviews:
    result = one_shot_chain.invoke({"text": r})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        print(f"Skipping invalid output: {result}")
        continue  # skip this review
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    one_shot_results.append(cleaned_label)


Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Re

In [78]:
df_new.head(10)

,Binary_label,Category,source,text,is_synthetic
0,fake,Home_and_Kitchen,amazon,Air pressure was utter crap. I kept the lid o...,1
1,fake,Home_and_Kitchen,amazon,"Not what I expected, smaller than expected, bu...",1
2,fake,Home_and_Kitchen,amazon,We like a toaster that doesn't have the long h...,1
3,fake,Home_and_Kitchen,amazon,Love these pans they work great. The only prob...,1
4,fake,Home_and_Kitchen,amazon,I love the print on the back and the finish. I...,1
5,fake,Home_and_Kitchen,amazon,Bought this on a whim and it was the best purc...,1
6,fake,Home_and_Kitchen,amazon,"I admit I was leary of the suction, but I did ...",1
7,fake,Home_and_Kitchen,amazon,Purchased one as a gift and it was a very nice...,1
8,fake,Home_and_Kitchen,amazon,I wanted something to hold a bunch of pieces o...,1
9,fake,Home_and_Kitchen,amazon,I'm so happy I found this set and am very plea...,1


In [80]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def evaluate_model(df, results):
    # Get true labels from the dataframe
    #y_true = df["deceptive"].tolist()
    y_true = df["Binary_label"].tolist()
    y_pred = results

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', pos_label=None)
    conf_matrix = confusion_matrix(y_true, y_pred)

    return accuracy, f1, conf_matrix

In [81]:
accuracy_one_shot, f1_one_shot, conf_matrix_one_shot = evaluate_model(df_new, one_shot_results)

print("=" * 50)
print("ONE-SHOT PROMPTING RESULTS")
print(Config.model)
print("=" * 50)
print("Accuracy:", accuracy_one_shot)
print("\nF1 Score:", f1_one_shot)
print("\nConfusion Matrix:")
print(conf_matrix_one_shot)
print("=" * 50)

ONE-SHOT PROMPTING RESULTS
gemma:7b
Accuracy: 0.54

F1 Score: 0.49776176438475817

Confusion Matrix:
[[100 300]
 [ 68 332]]


## Few-Shot Prompting
Using multiple examples to guide the model's classification

In [82]:
llm = OllamaLLM(model=Config.model)

In [83]:
few_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template=Config.few_shot_prompt_template
)

# Create few-shot chain
few_shot_chain = few_shot_prompt | llm

In [84]:
print(Config.few_shot_prompt_template)


    You are an expert at detecting real and fake reviews.


    Task: Classify the following review of a product that is listed on amazon as either "real" or "fake". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording and lacks details or specificity.

    Here are some examples:

    Example 1:
    Review: "Great Watch. I have had this watch for a long time and it works great. I have one in my car and the other in my office. The watch itself is a great watch and is very easy to use. It is a perfect size for my wrist and is very easy to wear. I have used it with a portable watch and I can tell the watch is very well made. I have worn it with a watch band and it is very comfortable. I will buy another one for my phone. I would highly recommend this watch.Perfect for my kids. I love them.These are super lightweight and lightweight. I have used them for a couple of months and they are still working gr

In [85]:
# Classify reviews using few-shot prompting
# few_shot_results = []
# for r in reviews:
#     result = few_shot_chain.invoke({"text": r})
#     # Clean the output
#     label = result.strip().lower()
#     if " " in label:
#         label = label.split()[0]
#     label = label.strip('.,!?;:')
#     print(f"Raw output: {result} -> Cleaned: {label}")
#     few_shot_results.append(label)

In [87]:
# Classify each review
few_shot_results = []

for r in reviews:
    result = few_shot_chain.invoke({"text": r})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        print(f"Skipping invalid output: {result}")
        continue  # skip this review
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    few_shot_results.append(cleaned_label)


Raw output: Fake.

The review lacks specifics, general language and lacks a thorough analysis of the product's features or performance. -> Cleaned: fake
Raw output: Fake.

The review lacks details and specifics, uses very general language, and lacks a genuine analysis of the product. -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Fake.

The review lacks specifics, uses generic language, and lacks detailed analysis or personal experience. -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Fake.

The review lacks specifics, generic language and lacks detail. -> Cleaned: fake
Raw output: Fake.

The review lacks specifics, uses general language, and lacks a detailed analysis of the product's features or performance. -> Cleaned: fake
Raw output: Fake.

The review lacks specifics, generic language, and lacks a personal touch. -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Real -> C

In [88]:
accuracy_few_shot, f1_few_shot, conf_matrix_few_shot = evaluate_model(df_new, few_shot_results)

print("=" * 50)
print("FEW-SHOT PROMPTING RESULTS")
print(Config.model)
print("=" * 50)
print("Accuracy:", accuracy_few_shot)
print("\nF1 Score:", f1_few_shot)
print("\nConfusion Matrix:")
print(conf_matrix_few_shot)
print("=" * 50)

FEW-SHOT PROMPTING RESULTS
gemma:7b
Accuracy: 0.61

F1 Score: 0.5924525344358014

Confusion Matrix:
[[161 239]
 [ 73 327]]


## Compare All Approaches
Compare zero-shot, one-shot, and few-shot results

In [89]:
# Compare all approaches
comparison_df = pd.DataFrame({
    'Approach': ['Zero-Shot', 'One-Shot', 'Few-Shot'],
    'Accuracy': [accuracy, accuracy_one_shot, accuracy_few_shot],
    'F1 Score': [f1, f1_one_shot, f1_few_shot]
})

print("=" * 60)
print("COMPARISON OF ALL PROMPTING APPROACHES")
print("=" * 60)
print(comparison_df.to_string(index=False))
print("=" * 60)

# Find best approach
best_idx = comparison_df['Accuracy'].idxmax()
best_approach = comparison_df.loc[best_idx, 'Approach']
best_accuracy = comparison_df.loc[best_idx, 'Accuracy']
print(f"\nBest Approach: {best_approach} (Accuracy: {best_accuracy:.4f})")

COMPARISON OF ALL PROMPTING APPROACHES
 Approach  Accuracy  F1 Score
Zero-Shot   0.51875  0.404024
 One-Shot   0.54000  0.497762
 Few-Shot   0.61000  0.592453

Best Approach: Few-Shot (Accuracy: 0.6100)


## Results With context for categories


In [90]:
pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [92]:
from prompt_category_context import Config

In [93]:
print(Config.zero_shot_prompt_template)


    You are an expert at detecting fake and real reviews.

    Task: Classify the following review of a product that is listed on amazon as either "fake" or "real". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording.

    The products reviewed may belong to categories such as:
    Home and Kitchen, Sports and Outdoors, Electronics, Beauty, Clothing, Books, and similar retail categories.


    Review: "{text}"

    Classify this review as either "fake" or "real" (respond with only one word):


### Zero shot


In [95]:
prompt = PromptTemplate(
    input_variables=["text"], # the placeholder to be replaced based on prompt template
    template=Config.zero_shot_prompt_template
)

# Load small model
llm = OllamaLLM(model=Config.model)

# Create chain using pipe operator (modern LangChain syntax)
chain = prompt | llm

In [96]:
# Classify each review
results = []

for r in reviews:
    result = chain.invoke({"text": r})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        print(f"Skipping invalid output: {result}")
        continue  # skip this review
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    results.append(cleaned_label)


Raw output: Real -> Cleaned: real


Raw output: Real.

The review provides specific details about the product's size and materials, indicating that the reviewer has actually used the product. -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real.

The review uses personal language and expresses genuine enthusiasm, suggesting it is authentic. -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Rea

In [97]:
# Classify each review
results = []

for r in reviews:
    result = chain.invoke({"text": r})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        print(f"Skipping invalid output: {result}")
        continue  # skip this review
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    results.append(cleaned_label)


Raw output: Real.

The review provides specific details about the product's air pressure and lid durability, indicating genuine experience. -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned

In [98]:
# def clean_label(text):
#     return text.strip().lower().split()[0].strip('.,!?;:')
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def evaluate_model(df, results):
    # Get true labels from the dataframe
    #y_true = df["deceptive"].tolist()
    y_true = df["Binary_label"].tolist()
    y_pred = results

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', pos_label=None)
    conf_matrix = confusion_matrix(y_true, y_pred)

    return accuracy, f1, conf_matrix

In [99]:
accuracy, f1, conf_matrix = evaluate_model(df_new, results)

print("=" * 50)
print("ZERO-SHOT PROMPTING RESULTS WITH CONTEXT")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

ZERO-SHOT PROMPTING RESULTS WITH CONTEXT
Accuracy: 0.51125

F1 Score: 0.3914100880393015

Confusion Matrix:
[[ 27 373]
 [ 18 382]]


## One Shot

In [100]:
llm = OllamaLLM(model=Config.model)

In [101]:
print(Config.one_shot_prompt_template)


     You are an expert at detecting fake and real reviews.

    Task: Classify the following review of a product that is listed on amazon as either "real" or "fake". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording and lacks details or specificity.

    The products reviewed may belong to categories such as:
    Home and Kitchen, Sports and Outdoors, Electronics, Beauty, Clothing, Books, and similar retail categories.

    Example:
    Review: "Great Watch. I have had this watch for a long time and it works great. I have one in my car and the other in my office. The watch itself is a great watch and is very easy to use. It is a perfect size for my wrist and is very easy to wear. I have used it with a portable watch and I can tell the watch is very well made. I have worn it with a watch band and it is very comfortable. I will buy another one for my phone. I would highly recommend this watch.Perfe

In [102]:
one_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template = Config.one_shot_prompt_template
)
# Create one-shot chain
one_shot_chain = one_shot_prompt | llm

In [103]:
# Classify each review
one_shot_results = []

for r in reviews:
    result = one_shot_chain.invoke({"text": r})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        print(f"Skipping invalid output: {result}")
        continue  # skip this review
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    one_shot_results.append(cleaned_label)


Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Re

In [104]:
accuracy_one_shot, f1_one_shot, conf_matrix_one_shot = evaluate_model(df_new, one_shot_results)

print("=" * 50)
print("ONE-SHOT PROMPTING RESULTS WITH CONTEXT")
print("=" * 50)
print("Accuracy:", accuracy_one_shot)
print("\nF1 Score:", f1_one_shot)
print("\nConfusion Matrix:")
print(conf_matrix_one_shot)
print("=" * 50)

ONE-SHOT PROMPTING RESULTS WITH CONTEXT
Accuracy: 0.5625

F1 Score: 0.5062599188855581

Confusion Matrix:
[[ 90 310]
 [ 40 360]]


## Few Shot

In [105]:
llm = OllamaLLM(model=Config.model)

In [106]:
print(Config.one_shot_prompt_template)


     You are an expert at detecting fake and real reviews.

    Task: Classify the following review of a product that is listed on amazon as either "real" or "fake". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording and lacks details or specificity.

    The products reviewed may belong to categories such as:
    Home and Kitchen, Sports and Outdoors, Electronics, Beauty, Clothing, Books, and similar retail categories.

    Example:
    Review: "Great Watch. I have had this watch for a long time and it works great. I have one in my car and the other in my office. The watch itself is a great watch and is very easy to use. It is a perfect size for my wrist and is very easy to wear. I have used it with a portable watch and I can tell the watch is very well made. I have worn it with a watch band and it is very comfortable. I will buy another one for my phone. I would highly recommend this watch.Perfe

In [107]:
few_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template=Config.few_shot_prompt_template
)

# Create few-shot chain
few_shot_chain = few_shot_prompt | llm

In [108]:
# Classify each review
few_shot_results = []

for r in reviews:
    result = few_shot_chain.invoke({"text": r})
    
    label = result.strip().lower()
    
    # Keep only valid labels
    if "fake" in label:
        cleaned_label = "fake"
    elif "real" in label:
        cleaned_label = "real"
    else:
        print(f"Skipping invalid output: {result}")
        continue  # skip this review
    cleaned_label = cleaned_label.strip('.,!?;:')
    print(f"Raw output: {result} -> Cleaned: {cleaned_label}")
    few_shot_results.append(cleaned_label)


Raw output: Fake.

The review lacks specifics, generic language, and fails to provide useful information about the product's features or performance. -> Cleaned: fake
Raw output: Fake.

The review lacks details and specificity, and uses very general language. -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Fake.

The review lacks specific details and uses overly generic language. -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Fake -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Fake.

The review lacks specifics and demonstrates generic language, suggesting it is fabricated. -> Cleaned: fake
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real -> Cleaned: real
Raw output: Real

In [109]:
accuracy_few_shot, f1_few_shot, conf_matrix_few_shot = evaluate_model(df_new, few_shot_results)

print("=" * 50)
print("FEW-SHOT PROMPTING RESULTS WITH CONTEXT")
print("=" * 50)
print("Accuracy:", accuracy_few_shot)
print("\nF1 Score:", f1_few_shot)
print("\nConfusion Matrix:")
print(conf_matrix_few_shot)
print("=" * 50)

FEW-SHOT PROMPTING RESULTS WITH CONTEXT
Accuracy: 0.55

F1 Score: 0.5030061226884608

Confusion Matrix:
[[ 97 303]
 [ 57 343]]
